# 11 - Faster R-CNN Swin-T

The complete benchmark day for Faster R-CNN with a Swin-T-FPN backbone: dataset preparation, the model
environment, the GPU adapter smoke gate, two-stage LR search, final training,
and evaluation - in that order, from one run.

Leave `START = False` to review every stage's contract first. Set it to `True`
and run all cells to start. Every stage is idempotent or resumable, so after a
disconnect you reopen this same notebook and run all cells again.

Training refuses to begin until the adapter smoke gate stores a signed READY
record for this commit, environment, dataset track, and resolution. Official
validation is never used for tuning or model selection.


In [ ]:
DATASET_TRACK = "2class"
# False previews every stage contract without downloading, provisioning,
# or training. Set True and run all cells to start the real run; rerun the
# same notebook after a disconnect to resume wherever it stopped.
START = False
# True runs the opt-in baseline+tuned x seeds 17/42/3407 variance matrix
# instead of the headline tuned/seed-42 recipe. Six runs, reported separately.
FULL_MATRIX = False
# False runs against session storage, which is DELETED when the
# session ends. Only for smoke runs - never HPO or final training.
USE_GOOGLE_DRIVE = True


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git"
REPOSITORY_BRANCH = "main"
SMOKE_TEST = os.environ.get("SMOKE_TEST", "").lower() in {"1", "true", "yes"}

# The only logic a notebook still owns: make `src` importable. Everything after
# this line - Git state, platform detection, paths, dependency policy - lives in
# src/notebook_bootstrap.py so all notebooks behave identically.
_override = os.environ.get("BENCHMARK_REPO_ROOT")
_candidates = (
    [Path(_override).expanduser()]
    if _override
    else [
        Path.cwd(),
        *Path.cwd().parents,
        Path("/content/aerial-object-detection-benchmark"),
        Path("/kaggle/working/aerial-object-detection-benchmark"),
    ]
)
REPO_PATH = next(
    (
        candidate.resolve()
        for candidate in _candidates
        if (candidate / "src" / "notebook_bootstrap.py").is_file()
    ),
    None,
)
if REPO_PATH is None:
    _host = (
        Path("/content")
        if Path("/content").is_dir()
        else Path("/kaggle/working")
        if Path("/kaggle/working").is_dir()
        else None
    )
    if _host is None:
        raise RuntimeError(
            "Run this notebook from the repository, or set BENCHMARK_REPO_ROOT "
            "to an existing clone."
        )
    REPO_PATH = (_host / "aerial-object-detection-benchmark").resolve()
    REPO_PATH.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_PATH)],
        check=True,
    )
sys.path.insert(0, str(REPO_PATH))

from src.notebook_bootstrap import bootstrap_notebook

bootstrap = bootstrap_notebook(
    REPO_PATH,
    requirements_file='requirements-hpo-colab.txt',
    use_google_drive=USE_GOOGLE_DRIVE,
    smoke_test=SMOKE_TEST,
)
notebook_environment = bootstrap.environment
REPO_PATH = notebook_environment.repository_root
DRIVE_ROOT = notebook_environment.artifact_root
NOTEBOOK_PLATFORM = notebook_environment.platform
print(bootstrap.summary())


In [ ]:
from src.workflows.model_pipeline import run_model_pipeline

MODEL_ID = "faster_rcnn_swin_t"
result = run_model_pipeline(
    REPO_PATH,
    DRIVE_ROOT,
    MODEL_ID,
    DATASET_TRACK,
    # A smoke run must reach the real pipeline - that is what proves the
    # notebook is wired to it - but must never start an expensive stage,
    # whatever the parameter cell above says.
    start=START and not SMOKE_TEST,
    full_matrix=FULL_MATRIX,
)
result
